In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import zipfile

from google.colab import files

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
# ============================================================
# EXTRACT ZIP FILE
# ============================================================

zip_path = "/content/archive (3).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content")

print("Dataset Extracted Successfully")

FileNotFoundError: [Errno 2] No such file or directory: '/content/archive (3).zip'

In [ ]:
# ============================================================
# DATASET PATHS
# ============================================================

train_dir = "/content/train"
test_dir  = "/content/test"

In [ ]:
print(os.listdir(train_dir))
print(os.listdir(test_dir))

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

# Define image size and batch size
IMG_SIZE = (48,48)
BATCH_SIZE = 64

# Load training dataset
# Images are resized to 48x48 grayscale
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    color_mode="grayscale",
    batch_size=BATCH_SIZE
)

# Load testing dataset
# Images are resized to 48x48 grayscale
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    color_mode="grayscale",
    batch_size=BATCH_SIZE
)

In [ ]:
# ============================================================
# DATASET INFORMATION
# ============================================================

# Display the total number of emotion classes
print("Number of Classes:", len(class_names))

# Display all emotion class names
print("Classes:", class_names)

In [ ]:
# ============================================================
# CLASS NAMES
# ============================================================

# Get emotion class names from the dataset
class_names = train_ds.class_names

print(class_names)

In [ ]:
# ============================================================
# DISPLAY SAMPLE IMAGES
# ============================================================

plt.figure(figsize=(12,12))

for images, labels in train_ds.take(1):

    for i in range(16):

        plt.subplot(4,4,i+1)

        plt.imshow(
            images[i].numpy().squeeze(),
            cmap="gray"
        )

        plt.title(class_names[labels[i]])

        plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# DATA AUGMENTATION
# ============================================================

# Apply random transformations to training images
# to improve model generalization
data_augmentation = keras.Sequential([

    # Randomly flip images horizontally
    layers.RandomFlip("horizontal"),

    # Randomly rotate images
    layers.RandomRotation(0.1),

    # Randomly zoom images
    layers.RandomZoom(0.1)

])

In [ ]:
# ============================================================
# MODEL 1 - BASIC CNN
# ============================================================

# Build a basic CNN model for emotion recognition
cnn_model = keras.Sequential([

    # Apply data augmentation
    data_augmentation,

    # Normalize pixel values
    layers.Rescaling(1./255),

    # First Convolution Block
    layers.Conv2D(32,3,activation='relu'),
    layers.MaxPooling2D(),

    # Second Convolution Block
    layers.Conv2D(64,3,activation='relu'),
    layers.MaxPooling2D(),

    # Third Convolution Block
    layers.Conv2D(128,3,activation='relu'),
    layers.MaxPooling2D(),

    # Convert feature maps into a feature vector
    layers.Flatten(),

    # Fully Connected Layer
    layers.Dense(128,activation='relu'),

    # Reduce overfitting
    layers.Dropout(0.5),

    # Output layer for 7 emotion classes
    layers.Dense(7,activation='softmax')
])

# Compile the model
cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
cnn_model.summary()

In [ ]:
# ============================================================
# TRAIN BASIC CNN
# ============================================================

history_cnn = cnn_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30
)

In [ ]:
# ============================================================
# MODEL 2 - DEEP CNN
# ============================================================

# Build a Deep CNN model for emotion recognition
deep_cnn = keras.Sequential([

    # Apply data augmentation
    data_augmentation,

    # Normalize pixel values
    layers.Rescaling(1./255),

    # First Convolution Block
    layers.Conv2D(64,3,padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Second Convolution Block
    layers.Conv2D(128,3,padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Third Convolution Block
    layers.Conv2D(256,3,padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Fourth Convolution Block
    layers.Conv2D(512,3,padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Convert feature maps to a feature vector
    layers.GlobalAveragePooling2D(),

    # Fully Connected Layer
    layers.Dense(512,activation='relu'),
    layers.Dropout(0.5),

    # Fully Connected Layer
    layers.Dense(256,activation='relu'),
    layers.Dropout(0.5),

    # Output layer for 7 emotion classes
    layers.Dense(7,activation='softmax')
])

# Compile the model
deep_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
deep_cnn.summary()

In [ ]:
# ============================================================
# TRAIN DEEP CNN
# ============================================================

history_deep = deep_cnn.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30
)

In [ ]:
# ============================================================
# CONVERT GRAYSCALE TO RGB
# ============================================================

# Convert grayscale images to RGB
# for MobileNetV2 input requirements
def gray_to_rgb(image, label):

    image = tf.image.grayscale_to_rgb(image)

    return image, label

# Apply conversion to training dataset
train_rgb = train_ds.map(gray_to_rgb)

# Apply conversion to testing dataset
test_rgb = test_ds.map(gray_to_rgb)

In [ ]:
# ============================================================
# MODEL 3 - MOBILENETV2
# ============================================================

# Load pretrained MobileNetV2 model
# without the final classification layer
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(48,48,3)
)

# Freeze pretrained layers
# to use MobileNetV2 as a feature extractor
base_model.trainable = False

# Build the MobileNetV2 emotion classifier
mobilenet_model = keras.Sequential([

    # Normalize pixel values
    layers.Rescaling(1./255),

    # Extract image features
    base_model,

    # Convert feature maps to a feature vector
    layers.GlobalAveragePooling2D(),

    # Reduce overfitting
    layers.Dropout(0.5),

    # Output layer for 7 emotion classes
    layers.Dense(7,activation='softmax')
])

# Compile the model
mobilenet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
mobilenet_model.summary()

In [ ]:
# ============================================================
# TRAIN MOBILENETV2
# ============================================================

history_mobile = mobilenet_model.fit(
    train_rgb,
    validation_data=test_rgb,
    epochs=15
)

In [ ]:
# ============================================================
# MODEL COMPARISON
# ============================================================

cnn_acc = max(history_cnn.history['val_accuracy'])
deep_acc = max(history_deep.history['val_accuracy'])
mobile_acc = max(history_mobile.history['val_accuracy'])

print("CNN Accuracy:", cnn_acc*100)
print("Deep CNN Accuracy:", deep_acc*100)
print("MobileNetV2 Accuracy:", mobile_acc*100)

plt.figure(figsize=(8,5))

plt.bar(
    ["CNN","Deep CNN","MobileNetV2"],
    [cnn_acc,deep_acc,mobile_acc]
)

plt.ylabel("Validation Accuracy")
plt.title("Model Comparison")

plt.show()

In [ ]:
# ============================================================
# MODEL ACCURACY COMPARISON
# ============================================================

import pandas as pd

# Create a table containing validation
# accuracy of all trained models
results = pd.DataFrame({
    "Model": ["CNN", "Deep CNN", "MobileNetV2"],
    "Validation Accuracy": [
        cnn_acc,
        deep_acc,
        mobile_acc
    ]
})

# Display the comparison table
print(results)

In [ ]:
# ============================================================
# ACCURACY COMPARISON OF ALL MODELS
# ============================================================

plt.figure(figsize=(10,6))

plt.plot(
    history_cnn.history['val_accuracy'],
    label='CNN'
)

plt.plot(
    history_deep.history['val_accuracy'],
    label='Deep CNN'
)

plt.plot(
    history_mobile.history['val_accuracy'],
    label='MobileNetV2'
)

plt.title("Validation Accuracy Comparison")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# TRAINING ACCURACY COMPARISON
# ============================================================

plt.figure(figsize=(10,6))

plt.plot(
    history_cnn.history['accuracy'],
    label='CNN'
)

plt.plot(
    history_deep.history['accuracy'],
    label='Deep CNN'
)

plt.plot(
    history_mobile.history['accuracy'],
    label='MobileNetV2'
)

plt.title("Training Accuracy Comparison")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

# Generate and display the classification report
# for all emotion classes.

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)

In [ ]:
# ============================================================
# UPLOAD IMAGE
# ============================================================

uploaded = files.upload()

In [ ]:
# ============================================================
# FACE DETECTION + PREPROCESSING
# ============================================================

image_path = list(uploaded.keys())[0]

# Read Image
img = cv2.imread(image_path)

# Convert to RGB for Display
img_rgb = cv2.cvtColor(
    img,
    cv2.COLOR_BGR2RGB
)

# Convert to Grayscale for Face Detection
gray = cv2.cvtColor(
    img,
    cv2.COLOR_BGR2GRAY
)

# Load Face Detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Detect Faces
faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(30,30)
)

# Check if Face Found
if len(faces) == 0:

    print("❌ No Face Detected")

else:

    # Take First Face
    x, y, w, h = faces[0]

    # Draw Rectangle Around Face
    cv2.rectangle(
        img_rgb,
        (x, y),
        (x+w, y+h),
        (0,255,0),
        2
    )

    # Crop Face
    face_gray = gray[y:y+h, x:x+w]

    # Resize to FER Input Size
    face_gray = cv2.resize(
        face_gray,
        (48,48)
    )

    # Show Original Image
    plt.figure(figsize=(6,6))
    plt.imshow(img_rgb)
    plt.title("Detected Face")
    plt.axis("off")
    plt.show()

    # Show Face Sent To Model
    plt.figure(figsize=(4,4))
    plt.imshow(face_gray, cmap="gray")
    plt.title("Face Sent To CNN")
    plt.axis("off")
    plt.show()

    # IMPORTANT:
    # Do NOT divide by 255 here
    # Model already contains Rescaling(1./255)

    cnn_input = np.expand_dims(
        face_gray,
        axis=(0,-1)
    ).astype("float32")

    print("Input Shape:", cnn_input.shape)

In [ ]:
# ============================================================
# CNN PREDICTION
# ============================================================

# Predict emotion probabilities using the CNN model
cnn_prediction = cnn_model.predict(cnn_input)

# Get the class with highest probability
cnn_class = np.argmax(cnn_prediction)

# Convert class index to emotion label
cnn_emotion = class_names[cnn_class]

# Get prediction confidence percentage
cnn_confidence = np.max(cnn_prediction) * 100

# Display prediction results
print("CNN Prediction")
print("Emotion:", cnn_emotion)
print("Confidence:", f"{cnn_confidence:.2f}%")

In [ ]:
prediction = cnn_model.predict(cnn_input)

for emotion, prob in zip(class_names, prediction[0]):
    print(f"{emotion}: {prob*100:.2f}%")

In [ ]:
# ============================================================
# DEEP CNN PREDICTION
# ============================================================

# Predict emotion probabilities using the Deep CNN model
deep_prediction = deep_cnn.predict(cnn_input)

# Get the class with highest probability
deep_class = np.argmax(deep_prediction)

# Convert class index to emotion label
deep_emotion = class_names[deep_class]

# Get prediction confidence percentage
deep_confidence = np.max(deep_prediction) * 100

# Display prediction results
print("\nDeep CNN Prediction")
print("Emotion:", deep_emotion)
print("Confidence:", f"{deep_confidence:.2f}%")

In [ ]:
# ============================================================
# MOBILENETV2 PREDICTION
# ============================================================

# Convert grayscale face image to RGB
# MobileNetV2 requires 3-channel input
mobile_face = cv2.cvtColor(
    face_gray.astype(np.float32),
    cv2.COLOR_GRAY2RGB
)

# Add batch dimension for model prediction
mobile_input = np.expand_dims(
    mobile_face,
    axis=0
)

# Predict emotion probabilities using MobileNetV2
mobile_prediction = mobilenet_model.predict(
    mobile_input
)

# Get the class with highest probability
mobile_class = np.argmax(
    mobile_prediction
)

# Convert class index to emotion label
mobile_emotion = class_names[
    mobile_class
]

# Get prediction confidence percentage
mobile_confidence = np.max(
    mobile_prediction
) * 100

# Display prediction results
print("\nMobileNetV2 Prediction")
print("Emotion:", mobile_emotion)
print("Confidence:", f"{mobile_confidence:.2f}%")

In [ ]:
# ============================================================
# COMPARISON TABLE
# ============================================================

print("\n")
print("="*50)
print("EMOTION PREDICTION COMPARISON")
print("="*50)

print(f"CNN          : {cnn_emotion:10s} {cnn_confidence:.2f}%")

print(f"Deep CNN     : {deep_emotion:10s} {deep_confidence:.2f}%")

print(f"MobileNetV2  : {mobile_emotion:10s} {mobile_confidence:.2f}%")

In [ ]:
# ============================================================
# CONFIDENCE GRAPH
# ============================================================

models = [
    "CNN",
    "Deep CNN",
    "MobileNetV2"
]

confidences = [
    cnn_confidence,
    deep_confidence,
    mobile_confidence
]

plt.figure(figsize=(8,5))

plt.bar(
    models,
    confidences
)

plt.ylabel("Confidence (%)")

plt.title(
    "Emotion Prediction Confidence"
)

plt.show()